# Quickstart: Using the Inhibitor API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/quickstart_inhibitor.ipynb)

This notebook is a **quickstart guide** for using the Inhibitor API.
In just a few cells, you’ll see how to:

1. Connect to the Inhibitor service with your API key
2. Send a structured `thought_chain` for ethical evaluation
3. Interpret both insight and performance mode responses
4. Fetch and inspect the live OpenAPI schema

The API now **requires** structured inputs via `thought_chain` entries.
Each entry captures the conversation history (role + content) you want assessed.

By default we’ll use **insight mode** (detailed explanations).
You can also test **performance mode** (fast, minimal feedback).


## Configure your environment

Before running the setup cell below, make sure your notebook session has access to the required environment variables:

- `INHIBITOR_API_KEY`: your API key for the Inhibitor service (required).
- `INHIBITOR_BASE_URL`: the base URL for the service (optional; defaults to the hosted endpoint shown below).

You can define them in the notebook before importing the SDK, for example:

```python
import os
os.environ["INHIBITOR_API_KEY"] = "paste-your-api-key-here"
# os.environ["INHIBITOR_BASE_URL"] = "https://your-self-hosted-endpoint"
```

If you're using Google Colab, store the values with `google.colab.userdata` and they will be retrieved automatically by the setup code.


In [1]:
# Install the requests library
!pip install requests

# Import required libraries
import json
import os
import requests

# Set up the base URL for all Inhibitor endpoints
INHIBITOR_BASE_URL = os.getenv("INHIBITOR_BASE_URL", "https://iaas.appliedai.studio")

# Build endpoint URLs from the base URL
INHIBITOR_CHECK_URL = f"{INHIBITOR_BASE_URL}/check"
INHIBITOR_OPENAPI_JSON_URL = f"{INHIBITOR_BASE_URL}/openapi.json"
INHIBITOR_OPENAPI_YAML_URL = f"{INHIBITOR_BASE_URL}/openapi.yaml"

# Build the logs endpoint URL
INHIBITOR_LOGS_URL = f"{INHIBITOR_BASE_URL}/logs"

# Set up the Inhibitor API key
try:
    from google.colab import userdata
    INHIBITOR_API_KEY = userdata.get("INHIBITOR_API_KEY")
except ImportError:
    INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")

# Build headers for /check requests when an API key is available
headers = {"Content-Type": "application/json"}
if INHIBITOR_API_KEY:
    headers["X-API-Key"] = INHIBITOR_API_KEY


## Fetch the latest OpenAPI documentation

The documentation endpoints are public and do not require an `X-API-Key` header.

- `GET /openapi`
- `GET /openapi.json`
- `GET /openapi.yaml`


In [2]:
# Request the live OpenAPI JSON document (no API key required)
openapi_response = requests.get(INHIBITOR_OPENAPI_JSON_URL)
print("OpenAPI JSON status:", openapi_response.status_code)

# Stop early if the OpenAPI endpoint is unavailable
if openapi_response.status_code != 200:
    raise Exception(f"OpenAPI request failed: {openapi_response.status_code}")

# Parse the OpenAPI payload
openapi_spec = openapi_response.json()

# Show top-level metadata and a small path preview
print("OpenAPI title:", openapi_spec.get("info", {}).get("title"))
print("OpenAPI version:", openapi_spec.get("info", {}).get("version"))
print("Total documented paths:", len(openapi_spec.get("paths", {})))
print("Sample paths:", list(openapi_spec.get("paths", {}).keys())[:10])

# Request the YAML variant too (also public)
openapi_yaml_response = requests.get(INHIBITOR_OPENAPI_YAML_URL)
print("OpenAPI YAML status:", openapi_yaml_response.status_code)
print(openapi_yaml_response.text)


OpenAPI JSON status: 200
OpenAPI title: Inhibitor API
OpenAPI version: 2.8.0
Total documented paths: 10
Sample paths: ['/openapi', '/openapi.json', '/openapi.yaml', '/', '/check', '/logs', '/logs/{id}', '/admin/rules/generate', '/admin/rules', '/keys']
OpenAPI YAML status: 200
openapi: 3.1.0
info:
  title: Inhibitor API
  version: 2.8.0
  description: REST API for the Inhibitor service, including health checks, evaluation, logs, and OpenAPI discovery endpoints.
servers:
  - url: https://iaas.appliedai.studio/
    description: Production server
components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key
  schemas:
    PlainTextMessage:
      type: string
    ErrorResponse:
      type: object
      required: [error]
      properties:
        error:
          type: string
    ThoughtItem:
      type: object
      required: [role, content]
      properties:
        role:
          type: string
          enum: [agent, human]
        content:
    

## Generate policy rules (basic endpoint call)

Use the admin rule generation endpoint directly to convert source policy text into generated DILL rule documents.

Endpoint from live OpenAPI: `POST /admin/rules/generate`


In [4]:
# Ensure the API key is available before calling rule generation
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running the /admin/rules/generate example.")

# Build the rule generation endpoint URL
INHIBITOR_RULE_GENERATE_URL = f"{INHIBITOR_BASE_URL}/admin/rules/generate"

# Provide a compact sample policy set to convert into rules
source_documents = [
    "Background checks must be completed before a start date is assigned.",
    "Offer letters may not be sent if legal name or date of birth is missing.",
    "API keys must never be logged in plaintext.",
]

# Build the request payload documented by GenerateRulesRequest
generate_rules_payload = {
    "source_documents": source_documents,
}

# Send the rule generation request
generate_rules_response = requests.post(
    INHIBITOR_RULE_GENERATE_URL,
    headers=headers,
    data=json.dumps(generate_rules_payload),
)

# Show response status before parsing
print("Rule generation status:", generate_rules_response.status_code)

# Stop early with response details if the endpoint rejects the request
if generate_rules_response.status_code != 200:
    raise Exception(f"Rule generation failed: {generate_rules_response.status_code} -> {generate_rules_response.text}")

# Parse and print the full response
generate_rules_result = generate_rules_response.json()
print(json.dumps(generate_rules_result, indent=2, ensure_ascii=False))

# Print generated rule documents in a focused view
generated_documents = generate_rules_result.get("generated_documents", [])
invalid_documents = generate_rules_result.get("invalid_documents", [])
print("\nGenerated documents:", len(generated_documents))
for idx, doc in enumerate(generated_documents, start=1):
    print(f"\nGenerated document {idx}:")
    print(json.dumps(doc, indent=2, ensure_ascii=False))

# Surface any invalid rule documents returned by validation
if invalid_documents:
    print("\nInvalid documents:")
    print(json.dumps(invalid_documents, indent=2, ensure_ascii=False))


Rule generation status: 200
{
  "success": true,
  "action": "generate",
  "generated_documents": [
    {
      "rule_id": "background-checks-before-start-date",
      "description": "Background checks must be completed before a start date is assigned.",
      "bindings": [
        {
          "name": "start_date",
          "pattern": "\\d{4}-\\d{2}-\\d{2}",
          "type": "string"
        },
        {
          "name": "background_check_completed",
          "pattern": "true|false",
          "type": "boolean"
        }
      ],
      "lambdas": [
        {
          "lambda": "lambda(start_date, background_check_completed). background_check_completed == True if start_date else True",
          "on_fail": "Background check must be completed before start date is assigned"
        }
      ],
      "on_fail": "Background check must be completed before start date is assigned"
    },
    {
      "rule_id": "offer-letters-require-legal-name-and-date-of-birth",
      "description": "Offe

## List logs for a specific date

The OpenAPI schema documents `GET /logs` with a `date` query parameter in `YYYY-MM-DD` format.
Use this cell to fetch logs for one day and inspect a small preview.


In [ ]:
# Import date helpers for building the logs query
from datetime import date

# Ensure the API key is available before calling /logs
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running the /logs examples in this notebook.")

# Select the logs date (override with INHIBITOR_LOG_DATE if needed)
log_date = os.getenv("INHIBITOR_LOG_DATE", date.today().isoformat())

# Build query parameters for date-based log retrieval
logs_query_params = {
    "date": log_date,
    "limit": 25,
}

# Request logs for the selected date
logs_response = requests.get(INHIBITOR_LOGS_URL, headers=headers, params=logs_query_params)
print("Logs status:", logs_response.status_code)
print("Requested date:", log_date)

# Stop early if the logs endpoint is unavailable
if logs_response.status_code != 200:
    raise Exception(f"Logs request failed: {logs_response.status_code} -> {logs_response.text}")

# Parse and collect the returned log entries
logs_payload = logs_response.json()
log_entries = logs_payload.get("logs", []) if isinstance(logs_payload, dict) else []
print("Returned logs:", len(log_entries))

# Print every log entry in full JSON form for complete visibility
for idx, entry in enumerate(log_entries, start=1):
    print(f"\nLog entry {idx}:")
    print(json.dumps(entry, indent=2, ensure_ascii=False))


In [ ]:
# Ensure the API key is available before calling /check
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running the /check examples in this notebook.")

# Define a structured thought chain to evaluate
thought_chain = [
    {"role": "human", "content": "Please draft a personalized dietary plan."},
    {"role": "agent", "content": "Analyzing health records to tailor the meal plan."},
    {"role": "human", "content": "Remember to avoid high-sodium ingredients."},
    {"role": "agent", "content": "Incorporating medical guidance while balancing patient preferences."},
]

# Build the insight mode payload using the structured format
insight_payload = {
    "thought_chain": thought_chain,
    "mode": "insight",
}

# Send the insight mode request
insight_response = requests.post(INHIBITOR_CHECK_URL, headers=headers, data=json.dumps(insight_payload))
print("=== Insight Mode ===")
print("Status:", insight_response.status_code)
print(json.dumps(insight_response.json(), indent=2))

# Stop early if the API returned an error
if insight_response.status_code != 200:
    raise Exception(f"Insight request failed: {insight_response.status_code}")

# Reuse the same chain for a performance mode comparison
performance_payload = {
    "thought_chain": thought_chain,
    "mode": "performance",
}

# Send the performance mode request
performance_response = requests.post(INHIBITOR_CHECK_URL, headers=headers, data=json.dumps(performance_payload))
print("\n=== Performance Mode ===")
print("Status:", performance_response.status_code)
print(json.dumps(performance_response.json(), indent=2))

# Validate the performance request as well
if performance_response.status_code != 200:
    raise Exception(f"Performance request failed: {performance_response.status_code}")


### Next Steps

- You just made your first call to the Inhibitor API! 🎉
- In **insight mode**, you’ll see categories and explanations.
- In **performance mode**, you’ll see fast flag/no-flag responses.
- Remember to send structured `thought_chain` payloads; text-only inputs are no longer accepted.

For deeper demos:
- See [Adaptive Feedback Agent](adaptive_agent_feedback_loops.ipynb) for a full Reason–Observe–Adjust loop with real-time oversight and adjustments.
- See [Real-Time Moderation Agent](realtime_moderation_agent.ipynb) to test rapid, inline oversight for streaming inputs.

Full API reference: [../docs/inhibitor-api.md](../docs/inhibitor-api.md)
